In [16]:
import pandas as pd
import json, os
import re


data_path = "DepMap/"

rna_expression_df = pd.read_csv(
    data_path + "Expression_Public_25Q3_subsetted.csv",
    index_col=0, low_memory=False
)
proteomics_df = pd.read_csv(
    data_path + "Harmonized_MS_CCLE_Gygi_subsetted.csv",
    index_col=0, low_memory=False
)
drug_response_df = pd.read_csv(
    data_path + "Drug_sensitivity_AUC_(PRISM_Repurposing_Secondary_Screen)_subsetted.csv",
    index_col=0, low_memory=False
)

os.makedirs("artifacts", exist_ok=True)
info = {
    "DepMap_release": "25Q3",
    "Files_used": [
        "Expression_Public_25Q3_subsetted.csv",
        "Harmonized_MS_CCLE_Gygi_subsetted.csv",
        "Drug_sensitivity_AUC_(PRISM_Repurposing_Secondary_Screen)_subsetted.csv"
        
    ]
}

os.makedirs("artifacts/aligned", exist_ok=True)
os.makedirs("artifacts/metadata", exist_ok=True)

with open("artifacts/metadata/dataset_info.json", "w") as f:
    json.dump(info, f, indent=2)

print("RNA shape:", rna_expression_df.shape)
print("Proteomics shape:", proteomics_df.shape)
print("Drug response shape:", drug_response_df.shape)

print("\nRNA example:")
display(rna_expression_df.iloc[:5, :5])
print("\nProteomics example:")
display(proteomics_df.iloc[:5, :5])
print("\nDrug response example:")
display(drug_response_df.iloc[:5, :5])


RNA shape: (1699, 19218)
Proteomics shape: (375, 12564)
Drug response shape: (480, 1456)

RNA example:


,cell_line_display_name,lineage_1,lineage_2,lineage_3,lineage_6
depmap_id,,,,,
ACH-001270,127399,Soft Tissue,Synovial Sarcoma,Synovial Sarcoma,NaN
ACH-002680,170MGBA,CNS/Brain,Diffuse Glioma,Glioblastoma,NaN
ACH-001438,1777NRPMET,Testis,Non-Seminomatous Germ Cell Tumor,Embryonal Carcinoma,NaN
ACH-002401,21MT2,Breast,Invasive Breast Carcinoma,Breast Invasive Ductal Carcinoma,HER2+
ACH-002399,21NT,Breast,Invasive Breast Carcinoma,Breast Invasive Ductal Carcinoma,HER2+



Proteomics example:


,cell_line_display_name,lineage_1,lineage_2,lineage_3,lineage_6
depmap_id,,,,,
ACH-000248,AU565,Breast,Invasive Breast Carcinoma,Invasive Breast Carcinoma,HER2+
ACH-000536,BT20,Breast,Invasive Breast Carcinoma,Breast Invasive Ductal Carcinoma,Basal A TNBC
ACH-000288,BT549,Breast,Invasive Breast Carcinoma,"Breast Invasive Carcinoma, NOS",Basal B TNBC
ACH-000212,CAL120,Breast,Invasive Breast Carcinoma,Invasive Breast Carcinoma,Basal B TNBC
ACH-000856,CAL51,Breast,Invasive Breast Carcinoma,Invasive Breast Carcinoma,Basal B TNBC



Drug response example:


,cell_line_display_name,lineage_1,lineage_2,lineage_3,lineage_6
depmap_id,,,,,
ACH-000973,639V,Bladder/Urinary Tract,Urethral Cancer,Urethral Urothelial Carcinoma,NaN
ACH-001016,BECKER,CNS/Brain,Diffuse Glioma,Astrocytoma,NaN
ACH-000927,BT474,Breast,Invasive Breast Carcinoma,Breast Invasive Ductal Carcinoma,"ER+, PR+, HER2+"
ACH-000288,BT549,Breast,Invasive Breast Carcinoma,"Breast Invasive Carcinoma, NOS",Basal B TNBC
ACH-000212,CAL120,Breast,Invasive Breast Carcinoma,Invasive Breast Carcinoma,Basal B TNBC


In [17]:
def audit_missing(df, name):
    n = df.shape[0] * df.shape[1]
    miss = int(df.isna().sum().sum())
    print(f"{name}: shape={df.shape}, total={n:,}, missing={miss:,} ({100*miss/n:.3f}%)")
    return miss, n

print("RAW TABLES")
audit_missing(rna_expression_df, "RNA raw")
audit_missing(proteomics_df, "Proteomics raw")
audit_missing(drug_response_df, "Drug raw")



RAW TABLES
RNA raw: shape=(1699, 19218), total=32,651,382, missing=3,171 (0.010%)
Proteomics raw: shape=(375, 12564), total=4,711,500, missing=1,309,394 (27.791%)
Drug raw: shape=(480, 1456), total=698,880, missing=302,991 (43.354%)


(302991, 698880)

In [18]:
# normalise indices & drop metadata columns if present
for df in [rna_expression_df, proteomics_df, drug_response_df]:
    df.index = df.index.astype(str)

META_COLS = [
    "cell_line_display_name", "lineage_1", "lineage_2",
    "lineage_3", "lineage_4", "lineage_6"
]
rna_expression_df = rna_expression_df.drop(columns=META_COLS, errors="ignore")
proteomics_df    = proteomics_df.drop(columns=META_COLS, errors="ignore")
drug_response_df = drug_response_df.drop(columns=META_COLS, errors="ignore")


In [19]:
# Map UniProt/isoforms → gene symbols and collapse duplicate genes

def extract_gene(col: str) -> str:
    m = re.search(r"\(([^)]+)\)", str(col))
    return m.group(1).strip() if m else str(col).strip()

# Rename Proteomics columns to gene symbols 
proteomics_df.columns = [extract_gene(c) for c in proteomics_df.columns]

# Ensure numeric
proteomics_df = proteomics_df.apply(pd.to_numeric, errors="coerce")

# Collapse duplicate gene symbols by mean
proteomics_df = proteomics_df.T.groupby(level=0, sort=False).mean().T

# Drop any columns that became all-NaN after coercion
proteomics_df = proteomics_df.dropna(axis=1, how="all")

# Sanity check
assert not proteomics_df.columns.duplicated().any(), "Unexpected duplicate gene columns remain in Proteomics."


In [20]:
# Quantifying missingness

def missing_report(df, name):
    missing_total = df.isna().sum().sum()
    total_values = df.shape[0] * df.shape[1]
    pct = 100 * missing_total / total_values
    print(f"{name}: {missing_total:,} / {total_values:,} missing ({pct:.4f}%)")
    return missing_total, total_values, pct

_ = missing_report(rna_expression_df, "RNA")
_ = missing_report(proteomics_df, "Proteomics")
_ = missing_report(drug_response_df, "Drug response")


RNA: 0 / 32,641,188 missing (0.0000%)
Proteomics: 1,228,773 / 4,547,250 missing (27.0223%)
Drug response: 302,075 / 696,000 missing (43.4016%)


# **Evaluation of Missing Data**

| Dataset            | Total Values |   Missing |    % Missing |
| :----------------- | -----------: | --------: | -----------: |
| **RNA Expression** |   32,641,188 |         0 | **0.0000 %** |
| **Proteomics**     |    4,547,250 | 1,228,773 |  **27.02 %** |
| **Drug Response**  |      696,000 |   302,075 |  **43.40 %** |

The table above summarises the proportion of missing data in each omics dataset after initial preprocessing.
These differences are **biologically and experimentally expected**, rather than signs of corruption or data-loading errors.

# **RNA Expression**

The RNA-seq dataset is **essentially complete**, with *no missing values detected* (0.0000 %).
This indicates that gene expression measurements are consistently available for nearly all cell lines and genes, so **no imputation or filtering** is needed at this stage. The RNA matrix can safely be used as-is for modelling.


# **Proteomics**

Approximately **27 %** of protein quantifications are missing, which is **typical** for large-scale mass-spectrometry (MS) data.
These missing values arise because not all proteins are detected in every cell line especially those of low abundance or weak ionisation efficiency.
This pattern is expected and will be handled later through **imputation strategies** or filtering of highly sparse proteins.


# **Drug Response**

Roughly **43 %** of drug–cell line response measurements are missing.
This missingness is **by design**, as not all drugs in the PRISM screen are tested across all cell lines.
These missing entries represent **untested pairs**, not measurement errors, and will be automatically excluded when focusing on individual drugs during downstream prediction experiments.




In [21]:
# See where missingness is (per feature / per cell line)

def show_missing(df, name, feature_label):
    # Missing by feature (columns)
    col_missing = df.isna().sum().sort_values(ascending=False)
    print(f"\nTop 10 features ({feature_label}) with most missing in {name}:")
    display(col_missing.head(10).rename("missing_count"))

    # Missing by cell line (rows)
    row_missing = df.isna().sum(axis=1).sort_values(ascending=False)
    print(f"\nTop 10 cell lines with most missing in {name}:")
    display(row_missing.head(10).rename("missing_count"))

show_missing(rna_expression_df, "RNA", "genes")
show_missing(proteomics_df, "Proteomics", "proteins")
show_missing(drug_response_df, "Drug response", "drugs")



Top 10 features (genes) with most missing in RNA:


TSPAN6      0
SLC25A33    0
PIK3CD      0
ZNF274      0
CXXC5       0
CLSTN1      0
NMUR1       0
DNAI2       0
DSCAM       0
ZNF584      0
Name: missing_count, dtype: int64


Top 10 cell lines with most missing in RNA:


depmap_id
ACH-001270    0
ACH-000890    0
ACH-000155    0
ACH-000677    0
ACH-000384    0
ACH-000985    0
ACH-000252    0
ACH-000366    0
ACH-000341    0
ACH-001702    0
Name: missing_count, dtype: int64


Top 10 features (proteins) with most missing in Proteomics:


...

depmap_id
ACH-000649    4204
ACH-000842    4204
ACH-000929    4204
ACH-000383    4204
ACH-000012    4204
ACH-000097    4204
ACH-000601    4204
ACH-000395    4204
ACH-000805    4204
ACH-000883    4145
Name: missing_count, dtype: int64


Top 10 features (drugs) with most missing in Drug response:


NELARABINE (BRD:BRD-K84466663-001-05-4)                          478
SODIUM-TANSHINONE-II-A-SULFONATE (BRD:BRD-K84798689-236-02-9)    477
KY02111 (BRD:BRD-K13302470-001-02-7)                             477
RG108 (BRD:BRD-K89391146-001-08-0)                               475
BAN-ORL-24 (BRD:BRD-K47049295-300-01-5)                          475
TILMICOSIN (BRD:BRD-K26634012-001-04-9)                          475
POMALIDOMIDE (BRD:BRD-A12994259-001-02-1)                        475
METOPROLOL (BRD:BRD-A03623303-045-09-5)                          474
ETOFYLLINE-CLOFIBRATE (BRD:BRD-K49840922-001-05-1)               474
METOXIBUTROPATE (BRD:BRD-A71725768-001-01-3)                     474
Name: missing_count, dtype: int64


Top 10 cell lines with most missing in Drug response:


depmap_id
ACH-000535    1108
ACH-000274    1064
ACH-000688     982
ACH-000537     979
ACH-000429     969
ACH-000311     961
ACH-000758     956
ACH-000736     952
ACH-000169     946
ACH-000480     930
Name: missing_count, dtype: int64


# **Summary of Data Quality and Missingness Across Datasets**

The three datasets **RNA expression**, **proteomics**, and **drug response**  were examined to identify missingness patterns across both features (genes, proteins, drugs) and samples (cell lines).
The findings confirm that the missing data patterns are biologically expected and consistent with the design and limitations of each experimental assay.

# **RNA Expression Data**

The RNA expression dataset shows **no missing values** across any features or cell lines.
The top genes (e.g. *TSPAN6*, *SLC25A33*, *PIK3CD*, *ZNF274*) and all listed cell lines (e.g. *ACH-001270*, *ACH-000890*) have **zero missing entries**.

**Interpretation:**
This confirms that the transcriptomic data are fully complete and reliable.
Every cell line has full gene coverage, meaning no imputation or cleaning is required.
The RNA dataset can therefore serve as a stable and high-quality reference modality for downstream multi-omics integration.

# **Proteomics Data**

The proteomics dataset displays **moderate missingness (~27%)**, concentrated among specific proteins and cell lines.
Some proteins such as *SLAMF1*, *TPH1*, *SCN5A*, *ENTREP3*, *ERVK-7*, and *KLK3* are undetected in a large number of samples (missing in 366–367 cell lines).
Similarly, several cell lines including *ACH-000649*, *ACH-000842*, *ACH-000929*, and *ACH-000383* have thousands of missing protein measurements, suggesting limited proteomic coverage in those specific samples.

**Interpretation:**
This is **typical for large-scale mass-spectrometry proteomics**, where low-abundance proteins or challenging samples often result in incomplete detection.
These missing values are not errors but reflect instrument sensitivity and peptide identification thresholds.
They will later be addressed using **imputation or filtering** strategies during model preparation.


# **Drug Response Data**

The drug response dataset exhibits **substantial missingness (~43%)**, primarily due to experimental design.
Several drugs such as *Nelarabine*, *KY02111*, *RG108*, and *BAN-ORL-24* have been tested only in a subset of cell lines (missing in 470–480 lines).
On the other hand, certain cell lines like *ACH-000535*, *ACH-000274*, and *ACH-000688* have missing results for over a thousand drugs, indicating limited testing coverage.

**Interpretation:**
This missingness is **intentional and expected** within the PRISM drug-screen setup, where not every compound is screened against every cell line.
The missing entries correspond to **untested drug–cell line pairs**, not measurement failures.
Each drug will later be modelled separately using only the subset of lines for which measurements exist.


# **Overall Summary**

* **RNA Expression:** Complete data; no missing values.
* **Proteomics:** Moderate missingness (~27%), biologically expected due to detection limits.
* **Drug Response:** High missingness (~43%), expected due to selective screening design.

These results confirm that all datasets are of **high technical quality** and represent realistic experimental conditions.
Global cleaning or imputation is unnecessary; instead, missingness will be managed **selectively** in later modelling and imputation stages to preserve biological fidelity.


In [22]:
# Align by common cell lines and save aligned Parquets
rna_cells  = set(rna_expression_df.index)
prot_cells = set(proteomics_df.index)
drug_cells = set(drug_response_df.index)
common_cells = sorted(rna_cells & prot_cells & drug_cells)

print("Number of common cell lines:", len(common_cells))

rna_aligned  = rna_expression_df.loc[common_cells]
prot_aligned = proteomics_df.loc[common_cells]
drug_aligned = drug_response_df.loc[common_cells]

# Sanity checks
assert rna_aligned.index.equals(prot_aligned.index)
assert rna_aligned.index.equals(drug_aligned.index)

print("Aligned shapes:")
print("  RNA:", rna_aligned.shape)
print("  Proteomics:", prot_aligned.shape)
print("  Drug:", drug_aligned.shape)

print("\nAligned DepMap IDs:", list(rna_aligned.index[:5]))
display(rna_aligned.iloc[:3, :5])
display(prot_aligned.iloc[:3, :5])
display(drug_aligned.iloc[:3, :5])

# Save aligned matrices
rna_aligned.to_parquet("artifacts/aligned/rna_aligned.parquet")
prot_aligned.to_parquet("artifacts/aligned/prot_aligned.parquet")
drug_aligned.to_parquet("artifacts/aligned/drug_aligned.parquet")

# Build a canonical gene inventory
gene_inventory = {
    "rna_genes": list(map(str, rna_aligned.columns)),
    "prot_genes": list(map(str, prot_aligned.columns)),
    "shared_genes": sorted(set(rna_aligned.columns) & set(prot_aligned.columns)),
}

# Enrich dataset_info.json
info_path = "artifacts/metadata/dataset_info.json"
info = {}
if os.path.exists(info_path):
    with open(info_path, "r") as f:
        info = json.load(f)

info.update({
    "DepMap_release": "25Q3",
    "n_common_cells": len(common_cells),
    "shapes": {
        "rna_aligned":  list(rna_aligned.shape),
        "prot_aligned": list(prot_aligned.shape),
        "drug_aligned": list(drug_aligned.shape),
    },
    "feature_counts": {
        "rna_genes":   int(rna_aligned.shape[1]),
        "prot_genes":  int(prot_aligned.shape[1]),
        "shared_genes": len(gene_inventory["shared_genes"]),
    }
})

with open(info_path, "w") as f:
    json.dump(info, f, indent=2)

with open("artifacts/metadata/gene_inventory.json", "w") as f:
    json.dump(gene_inventory, f, indent=2)

print("\nSaved aligned matrices and updated dataset_info.json + gene_inventory.json")


Number of common cell lines: 248
Aligned shapes:
  RNA: (248, 19212)
  Proteomics: (248, 12126)
  Drug: (248, 1450)

Aligned DepMap IDs: ['ACH-000007', 'ACH-000008', 'ACH-000012', 'ACH-000014', 'ACH-000015']


,TSPAN6,TNMD,DPM1,SCYL3,FIRRM
depmap_id,,,,,
ACH-000007,3.693431,0.0,6.139474,3.419419,3.979266
ACH-000008,3.179122,0.0,7.356258,2.359482,4.380307
ACH-000012,5.432288,0.0,6.255909,2.157157,3.974786


,KANSL1L,SLC12A8,RBM47,IFT56,TSNARE1
depmap_id,,,,,
ACH-000007,NaN,0.369309,0.666620,-0.631528,NaN
ACH-000008,0.374598,NaN,0.109219,NaN,NaN
ACH-000012,NaN,0.581007,1.878433,NaN,NaN


,8-BROMO-CGMP (BRD:BRD-A00077618-236-07-6),NORETYNODREL (BRD:BRD-A00758722-001-04-9),PREDNISOLONE-ACETATE (BRD:BRD-A01643550-001-04-9),BETAMETHASONE (BRD:BRD-A02180903-001-04-5),MEPIVACAINE (BRD:BRD-A03216249-003-24-3)
depmap_id,,,,,
ACH-000007,NaN,0.903644,NaN,0.938537,NaN
ACH-000008,NaN,0.880756,NaN,NaN,NaN
ACH-000012,NaN,0.877278,0.93309,0.837461,NaN



Saved aligned matrices and updated dataset_info.json + gene_inventory.json



# **Data Alignment Across Modalities**

After loading and cleaning each omic dataset, all three matrices were aligned based on **common DepMap cell line identifiers**.
Only cell lines present in **RNA**, **proteomics**, and **drug response** data were retained, ensuring complete multi-omic correspondence across samples.

| Dataset           | Shape (rows × features) | Missing Values (%) |
| :---------------- | :---------------------- | :----------------- |
| **RNA**           | (248 × 19,212)          | 0.00%              |
| **Proteomics**    | (248 × 12,126)          | 27.31%             |
| **Drug Response** | (248 × 1,450)           | 43.08%             |

A total of **248 common cell lines** were retained after intersection, guaranteeing consistent indexing across all modalities.
The first few aligned samples (e.g. *ACH-000007*, *ACH-000008*, *ACH-000012*) demonstrate matching entries across gene expression, protein abundance, and drug sensitivity profiles.

**Interpretation:**
The alignment confirms that downstream analyses will operate on a **shared multi-omic feature space**.
This ensures each sample contributes data across all molecular layers for integrated modelling.
Minor missingness within proteomics and drug response features will be handled during imputation and model training, while RNA remains fully complete.


# Next, to understand how the incomplete measurements within the aligned datasets might affect downstream modelling, we analyse the nature of the missing values distinguishing between random (MAR) and systematic (MNAR) missingness.

# **Interpretation of Missingness Mechanisms (MAR vs MNAR)**

Understanding the mechanism behind missing values is crucial for selecting an appropriate imputation strategy.

# **RNA Expression**

The RNA-seq dataset shows **no detectable missingness**.
All cell lines have complete measurements across all genes, suggesting that **no missingness mechanism** needs to be modelled.
Any small gaps (if they appeared later due to merging) would likely be **Missing Completely at Random (MCAR)**, driven by technical noise rather than biological bias.

# **Proteomics**

The proteomics data exhibit **structured missingness** typical of large-scale mass-spectrometry experiments.
Low-abundance or hydrophobic proteins are often undetected in certain cell lines because of instrument sensitivity or ionisation efficiency.
This pattern reflects a **mixture of MAR and MNAR**:

* **MAR component:** some missingness depends on experimental conditions or batch effects (e.g. run-specific detection limits).
* **MNAR component:** proteins with low true abundance are systematically more likely to be missing.

**Interpretation:** imputation should account for this left-censoring behaviour for example, using intensity-aware or KNN-based methods rather than simple mean filling.

# **Drug Response**

Drug-response data are **intentionally incomplete**, since not every drug is screened across every cell line in the PRISM panel.
These missing entries are **Missing Completely by Design**, meaning they are not random nor related to measurement failure.
When modelling, each drug’s response will be analysed using only the subset of tested lines.

# **Summary**

| Omic               | Primary Missingness Mechanism | Notes                                   |
| :----------------- | :---------------------------- | :-------------------------------------- |
| **RNA Expression** | MCAR / none                   | Fully complete; negligible concern      |
| **Proteomics**     | Mixed MAR + MNAR              | Left-censoring and detection-limit bias |
| **Drug Response**  | Missing by design (MCAR-like) | Not all drugs tested on all lines       |

Overall, **proteomics** is the only modality where modelling missingness behaviour is essential.
This reasoning justifies performing an **imputation bake-off** in the next notebook to identify the most suitable method for handling MNAR-like proteomic gaps.

This addition closes your missingness exploration stage nicely and transitions cleanly into **Notebook 03 – Imputation Bake-Off**.
